In [1]:
#Load Libraries
library(Seurat)
library(glmGamPoi)
library(data.table)
library(dplyr)
library(ggplot2)
library(patchwork)
library(Matrix)
library(future)

#Set Options
options(future.globals.maxSize = 400000 * 1024^2) #for 400GB max size
plan("sequential")

#Set working directory
setwd("/storage1/fs1/jmillman/Active/DigitalTwin")

Loading required package: SeuratObject

Loading required package: sp

‘SeuratObject’ was built with package ‘Matrix’ 1.7.3 but the current
version is 1.7.4; it is recomended that you reinstall ‘SeuratObject’ as
the ABI for ‘Matrix’ may have changed


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t



Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following object is masked from ‘package:glmGamPoi’:

    vars


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘ggplot2’


The following object is masked from ‘package:glmGamPoi’:

    vars




# Integrate RNA Data

In [2]:
obj <- readRDS("checkpoints/Fetal/FetalIsletAtlas_RNAmerged.rds")
obj

An object of class Seurat 
43478 features across 258731 samples within 1 assay 
Active assay: RNA (43478 features, 2000 variable features)
 109 layers present: counts.cao_H26350, counts.cao_H26547, counts.cao_H27058, counts.cao_H27464, counts.cao_H27471, counts.cao_H27552, counts.cao_H27870, counts.cao_H27876, counts.cao_H27948, counts.goncalves_7+0, counts.goncalves_7+1, counts.goncalves_7+3, counts.goncalves_9+6, counts.ma_S113, counts.ma_S120, counts.ma_S129, counts.ma_S13, counts.ma_S14, counts.ma_S27, counts.ma_S35, counts.ma_S37, counts.ma_S57, counts.ma_S66, counts.ma_S76, counts.ma_S82, counts.ma_S85, counts.ma_S86, counts.ma_S88, counts.ma_S9, counts.ma_S91, counts.migliorini_S1, counts.migliorini_S2, counts.migliorini_S3, counts.migliorini_S4, counts.migliorini_S5, counts.migliorini_S6, counts.xu_emb1, counts.xu_emb2, counts.xu_emb3, counts.xu_emb4, counts.xu_emb5, counts.xu_emb7, counts.yu_W10, counts.yu_W11, counts.yu_W12, counts.yu_W14, counts.yu_W15, counts.yu_W16, counts

In [3]:
Idents(obj) <- 'dataset'
table(Idents(obj))


   cao_H26350    cao_H26547    cao_H27058    cao_H27464    cao_H27471 
          550          3051          2095           769          1629 
   cao_H27552    cao_H27870    cao_H27876    cao_H27948 goncalves_7+0 
          698         30667         17794          2812           154 
goncalves_7+1 goncalves_7+3 goncalves_9+6       ma_S113       ma_S120 
          231           625           386          7574          7196 
      ma_S129        ma_S13        ma_S14        ma_S27        ma_S35 
         2648          5101          5090          2665          4726 
       ma_S37        ma_S57        ma_S66        ma_S76        ma_S82 
         5152          1735          2639          5992          6294 
       ma_S85        ma_S86        ma_S88         ma_S9        ma_S91 
         2418          4463          4791          3837          3420 
migliorini_S1 migliorini_S2 migliorini_S3 migliorini_S4 migliorini_S5 
        13493         11769         10220         12966         11459 
migli

In [4]:
obj <- IntegrateLayers(
  object = obj, method = HarmonyIntegration,
  orig.reduction = "pca", new.reduction = "harmony",
  verbose = FALSE)

obj <- FindNeighbors(obj, reduction = "harmony", dims = 1:30, verbose = FALSE)
obj <- FindClusters(obj, resolution = 0.6, cluster.name = "harmony_clusters", verbose = FALSE)
obj <- RunUMAP(obj, reduction = "harmony", dims = 1:30, reduction.name = "umap.harmony", verbose = FALSE)

The `features` argument is ignored by `HarmonyIntegration`.
This message is displayed once per session.
Warning message:
“Quick-TRANSfer stage steps exceeded maximum (= 12936550)”
Warning message:
“Quick-TRANSfer stage steps exceeded maximum (= 12936550)”
Warning message:
“Quick-TRANSfer stage steps exceeded maximum (= 12936550)”
Warning message:
“The default method for RunUMAP has changed from calling Python UMAP via reticulate to the R-native UWOT using the cosine metric
To use Python UMAP via reticulate, set umap.method to 'umap-learn' and metric to 'correlation'
This message will be shown once per session”


In [5]:
obj[["RNA"]] <- JoinLayers(obj[["RNA"]])
saveRDS(obj, file="checkpoints/Fetal/FetalIsletAtlas_RNAintegrated.rds")

In [6]:
sessionInfo()

R version 4.5.1 (2025-06-13)
Platform: x86_64-pc-linux-gnu
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/atlas/libblas.so.3.10.3 
LAPACK: /usr/lib/x86_64-linux-gnu/atlas/liblapack.so.3.10.3;  LAPACK version 3.10.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: Etc/UTC
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] future_1.68.0      Matrix_1.7-4       patchwork_1.3.2    ggplot2_4.0.1     
 [5] dplyr_1.1.4        data.table_1.18.0  glmGamPoi_1.20.0   Seurat_5.4.0.9000 
 [9] SeuratObject_5.0.2 sp_2.2-0          

loaded via a namespace (and not attached):
  [1] RCo